# Evaluation Tool Demo

## What is the Evaluation Tool?

Evaluation Tool is a set of tools in the utilities of the RI. They've been created to handle evaluation and prediction with the utilization of cache.

This demo will walk through how the prediction function opperates and how it can be included in your test stage.

## Demo 1: Prediction

Evaluation has 2 steps under the hood. The first step is prediction which will be shown here.

## What is Prediction?

Prediction in this case is the output of computing a model against a dataset. This output has 2 pieces:

- A list of predictions.
- A list of batches of data used to compute each prediction.

This demo will use object detection to show how it works. The output will be:

- A list of predictions per image. Each prediction is shown as a detection target with a bounding box, a label, and a confidence score.
- A list of batched data. These will represent the input data for the image, the ground truth target data, and the metadata of the image being used.

In [ ]:
from jatic_ri.util.cache import JSONCache, SimpleRICacheOD
from jatic_ri.util.evaluation import EvaluationTool, SimpleDataLoader
from jatic_ri.object_detection.models import TorchvisionODModel
from jatic_ri.object_detection.datasets import CocoDetectionDataset
import os
import torch
import time

A portion of the 2017 coco dataset is provided here. It has been resized to be equal sized.

In [ ]:
coco_dataset = CocoDetectionDataset(
        root="../../tests/testing_utilities/example_data/coco_resized_val2017",
        ann_file="../../tests/testing_utilities/example_data/coco_resized_val2017/instances_val2017_resized_6.json",
    )

In [ ]:
torch_wrapper = TorchvisionODModel(model_name="fasterrcnn_resnet50_fpn_v2", device="cpu")

In [ ]:
model=torch_wrapper
model_id="fasterrcnn_resnet50_fpn_v2"
dataset=coco_dataset
dataset_id="coco_resized_val2017"

dataloader = SimpleDataLoader(dataset,2)

The evaluation tool has a caching mechanism for predictions and metric results.

To enable the caching mechanism, pass in a RICache class into Evaluation tool that meets the needs of the model type you are evaluating.

This class type takes a directory path str. If the directory does not exist, it creates the directory. If none passes in, it uses the current working directory as the cache path.

In [ ]:
current_dir = os.getcwd()
tmp_dir = os.path.join(current_dir, 'tmp_prediction')
evaluationtool = EvaluationTool(SimpleRICacheOD(cache_root_dir=tmp_dir))

To call the tool use:

In [ ]:
start_time = time.time()

pred, data = evaluationtool.predict(
    model=model,
    model_id=model_id,
    dataset=dataset,
    dataset_id=dataset_id,
    dataloader=dataloader,
    batch_size=2
)
end_time = time.time()
print(f"Elapsed time: {end_time - start_time:.2f} seconds")

Running the selected code again with the same configuration, you will notice a much faster execution thanks to the cached results.

In [ ]:
start_time = time.time()

pred, data = evaluationtool.predict(
    model=model,
    model_id=model_id,
    dataset=dataset,
    dataset_id=dataset_id,
    dataloader=dataloader,
    batch_size=2
)
end_time = time.time()
print(f"Elapsed time: {end_time - start_time:.2f} seconds")

## Demo 2: Evaluate

## What is Evaluation?

An evaluation takes the outputs of prediction and evaluates them against a given metric.
This demo will use object detection to show how it works. The output will be:

- A list of predictions per image. Each prediction is shown as a detection target with a bounding box, a label, and a confidence score.
- A list of batched datas. These will represent the input data for the image, the ground truth target data, and the metadata of the image being used.
- A metric evaluated based off of the prediction and target data.

In [ ]:
from jatic_ri.object_detection.metrics import map50_torch_metric_factory
metric_id = "map50_metric"
metric_wrapper = map50_torch_metric_factory()

tmp_dir = os.path.join(current_dir, 'tmp_evaluation')
evaluationtool = EvaluationTool(SimpleRICacheOD(cache_root_dir=tmp_dir))

In [ ]:
start_time = time.time()

results = evaluationtool.evaluate(
    model=torch_wrapper,
    model_id=model_id,
    dataset=dataset,
    dataset_id=dataset_id,
    dataloader=dataloader,
    metric=metric_wrapper,
    metric_id=metric_id,
    batch_size=2
)
end_time = time.time()
print(f"Elapsed time: {end_time - start_time:.2f} seconds")

Running the selected code again with the same configuration, you will notice a much faster execution thanks to the cached results.

In [ ]:
start_time = time.time()

results = evaluationtool.evaluate(
    model=torch_wrapper,
    model_id=model_id,
    dataset=dataset,
    dataset_id=dataset_id,
    dataloader=dataloader,
    metric=metric_wrapper,
    metric_id=metric_id,
    batch_size=2
)
end_time = time.time()
print(f"Elapsed time: {end_time - start_time:.2f} seconds")

## Demo 3: Compute Metric

Inside the evaluate function is a compute_metric function. 

In most cases, evaluation should handle the use cases of providing metrics. None the less, this demo will provide an example of using the tool outside of evaluate.

## What is a Metric?

A metric is a function that is used to measure the performance of a model during or after training. Metrics are essential for assessing the accuracy, effectiveness, or other quality aspects of a model.

Here, the metric function is treated as an object. That object has an update, compute, and reset aspect to it. Updates will be applied to each prediction and the target data associated with the prediction. Then using all of the updates, the metric will compute the efficiency of it's target variable like acuracy. It can also be reset when needed.

The output of this function is usually a dictionairy with a key (the name of the metric) and a value (usually in tensor format).

In [ ]:
tmp_dir = os.path.join(current_dir, 'tmp_metric_result')
evaluationtool = EvaluationTool(SimpleRICacheOD(cache_root_dir=tmp_dir))

In [ ]:
start_time = time.time()

metric_results = evaluationtool.compute_metric(
            metric=metric_wrapper,
            filename="Demo_Cache_ID.json",
            prediction=pred,
            data=data
        )
end_time = time.time()
print(f"Elapsed time: {end_time - start_time:.2f} seconds")

Running the selected code again with the same configuration, you will notice a much faster execution thanks to the cached results.

In [ ]:
start_time = time.time()

metric_results = evaluationtool.compute_metric(
            metric=metric_wrapper,
            filename="Demo_Cache_ID.json",
            prediction=pred,
            data=data
        )
end_time = time.time()
print(f"Elapsed time: {end_time - start_time:.2f} seconds")

## Demo 4: Caching in Action

The following demo will show how evaluate can leverage precomputed predictions and metrics to leverage a faster operation.

In [ ]:
evaluationtool = EvaluationTool(SimpleRICacheOD())
cache_file_metric = f"{model_id}_{dataset_id}_{metric_id}_2.json"

In [ ]:
start_time = time.time()
pred, data = evaluationtool.predict(
    model=model,
    model_id=model_id,
    dataset=dataset,
    dataset_id=dataset_id,
    dataloader=dataloader,
    batch_size=2
)
end_time = time.time()
print(f"Elapsed time: {end_time - start_time:.2f} seconds")

In [ ]:
start_time = time.time()

metric_results = evaluationtool.compute_metric(
            metric=metric_wrapper,
            filename=cache_file_metric,
            prediction=pred,
            data=data
        )
end_time = time.time()
print(f"Elapsed time: {end_time - start_time:.2f} seconds")

In [ ]:
start_time = time.time()

results = evaluationtool.evaluate(
    model=torch_wrapper,
    model_id=model_id,
    dataset=dataset,
    dataset_id=dataset_id,
    dataloader=dataloader,
    metric=metric_wrapper,
    metric_id=metric_id,
    batch_size=2
)
end_time = time.time()
print(f"Elapsed time: {end_time - start_time:.2f} seconds")